# 01. 데이터 수집

**목적**: 위키백과 / Wikivoyage / 외교부 0404에서 괌 여행 정보를 수집하여 `data/raw/*.md`로 저장.

**카테고리 6개**: 리조트 / 액티비티 / 식당 / 명소 / 교통 / 안전·날씨

**라이선스**: CC BY-SA (위키 시리즈) + 공공누리 1유형 (외교부)

**도구**: `WebBaseLoader` (S2-1 강의 표 5대 로더 중 하나)

In [1]:
import os
from pathlib import Path

from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader가 사이트에 보낼 식별 정보
# (위키미디어 User-Agent 정책 준수 + HTTP 헤더 ASCII 제한 준수)
os.environ["USER_AGENT"] = (
    "guam-family-chatbot/0.1 "
    "(educational project; https://github.com/owenkim0219/guam-family-chatbot)"
)

# 데이터 저장 경로 (practice/ 기준 한 단계 위)
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"저장 경로: {RAW_DIR.resolve()}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


저장 경로: C:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\data\raw


In [2]:
# 1차 데이터 소스: 동작 확인용 (가장 확실한 메인 페이지 3개)
# 카테고리는 페이지의 주된 주제로 할당. 추가 페이지는 동작 확인 후 보강.
SOURCES = [
    {
        "url": "https://ko.wikipedia.org/wiki/%EA%B4%8C",  # 한국어 위키백과 "괌"
        "category": "general",
        "title": "괌 (한국어 위키백과)",
    },
    {
        "url": "https://en.wikipedia.org/wiki/Guam",
        "category": "general",
        "title": "Guam (English Wikipedia)",
    },
    {
        "url": "https://en.wikivoyage.org/wiki/Guam",
        "category": "general",
        "title": "Guam (Wikivoyage)",
    },
]

print(f"1차 URL: {len(SOURCES)}개")
for s in SOURCES:
    print(f"  - [{s['category']}] {s['title']}")

1차 URL: 3개
  - [general] 괌 (한국어 위키백과)
  - [general] Guam (English Wikipedia)
  - [general] Guam (Wikivoyage)


In [3]:
# Cell 3: 첫 번째 URL만 로딩 (동작 확인용)
test_source = SOURCES[0]  # "괌 (한국어 위키백과)"

loader = WebBaseLoader(test_source["url"])
docs = loader.load()

print(f"로드된 Document 수: {len(docs)}")
print(f"첫 Document 글자 수: {len(docs[0].page_content)}")
print(f"메타데이터: {docs[0].metadata}")
print()
print("=== 본문 첫 300자 ===")
print(docs[0].page_content[:300])

로드된 Document 수: 1
첫 Document 글자 수: 12972
메타데이터: {'source': 'https://ko.wikipedia.org/wiki/%EA%B4%8C', 'title': '괌 - 위키백과, 우리 모두의 백과사전', 'language': 'ko'}

=== 본문 첫 300자 ===




괌 - 위키백과, 우리 모두의 백과사전































본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















보이기

















기부

계정 만들기

로그인










In [4]:
# Cell 4: 본문 영역만 파싱 (위키미디어 표준 ID 활용)
import bs4

# 위키백과의 본문 컨테이너 ID (위키미디어 표준 — multi-class 이슈 회피)
bs4_strainer = bs4.SoupStrainer(id="mw-content-text")

loader = WebBaseLoader(
    test_source["url"],
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

print(f"로드된 Document 수: {len(docs)}")
print(f"첫 Document 글자 수: {len(docs[0].page_content)}")
print(f"메타데이터: {docs[0].metadata}")
print()
print("=== 본문 첫 300자 ===")
print(docs[0].page_content[:300])

로드된 Document 수: 1
첫 Document 글자 수: 9560
메타데이터: {'source': 'https://ko.wikipedia.org/wiki/%EA%B4%8C'}

=== 본문 첫 300자 ===
이 문서는 영어 위키백과의 Guam 문서를 번역하여, 문서의 내용을 확장할 필요가 있습니다.중요한 번역 안내를 보려면 [펼치기]를 클릭하십시오.
신뢰성 있고 확인할 수 있는 출처가 제시되도록 번역하여 주십시오.
번역을 완료한 후에는 {{번역된 문서}} 틀을 토론창에 표기하여 저작자를 표시하여 주십시오.
문맥상 이해를 돕기 위해 관련 문서를 같이 번역해주시는 것이 좋습니다.
번역을 확장할 필요가 있는 내용이 포함된 다른 문서를 보고 싶으시다면 분류:번역 확장 필요 문서를 참고해주세요.
괌차모로어: Guåhån영어: Guam


국기



In [5]:
# Cell 5: 위키미디어 페이지의 잡 요소 정제 함수
import re
import requests
from langchain_core.documents import Document

# 위키미디어 표준 잡 요소 클래스 (위키백과 + Wikivoyage 공통)
REMOVE_SELECTORS = [
    ".ambox",           # 알림 박스 (번역 안내, stub 등)
    ".infobox",         # 인포박스 (국기, 인구 등 표 형태)
    ".navbox",          # 하단 네비게이션 박스
    ".thumb",           # 이미지 썸네일 캡션
    "sup.reference",    # 인용 표시 [1], [2] 등
    ".mw-editsection",  # "[편집]" 링크
    ".reflist",         # 참고문헌 목록
    ".hatnote",         # "다른 뜻은 ..." 안내
    ".article-status",  # Wikivoyage 페이지 등급 메시지(outline/usable/guide/star)
]


def clean_wiki_text(url: str) -> str:
    """위키미디어 페이지에서 잡 요소 제거 후 본문 텍스트만 반환."""
    headers = {"User-Agent": os.environ["USER_AGENT"]}
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()

    soup = bs4.BeautifulSoup(resp.text, "html.parser")
    content = soup.find(id="mw-content-text")
    if content is None:
        return ""

    # 잡 요소 직접 제거
    for selector in REMOVE_SELECTORS:
        for tag in content.select(selector):
            tag.decompose()

    # 단락(<p>)와 섹션 헤더(<h2>~<h4>)만 추출
    parts = []
    for tag in content.find_all(["p", "h2", "h3", "h4"]):
        text = tag.get_text(separator="").strip()    # ← 변경
        if text:
            parts.append(text)

    text = "\n\n".join(parts)

    # 후처리: a 태그 경계에서 발생하는 잔여 공백 정리
    text = re.sub(r"  +", " ", text)              # 연속 공백 → 1개
    text = re.sub(r"\(\s+", "(", text)            # 여는 괄호 직후 공백
    text = re.sub(r"\s+\)", ")", text)            # 닫는 괄호 직전 공백
    text = re.sub(r"\s+([,;:.])", r"\1", text)    # 구두점 앞 공백

    return text


# 동일한 URL로 정제 효과 검증
text = clean_wiki_text(test_source["url"])
print(f"정제 후 글자 수: {len(text)}")
print()
print("=== 정제된 본문 첫 500자 ===")
print(text[:500])

정제 후 글자 수: 4812

=== 정제된 본문 첫 500자 ===
괌(영어: Guam, 차모로어: Guåhan, 문화어: 괌섬)은 오세아니아의 미크로네시아 마리아나 제도에 있는 미국의 해외 속령이다. 차모로인들이 약 4000년 전에 이곳에 정착해 원주민이 되었다. 마리아나 제도에서 가장 큰 섬이며 또한 최남단에 있는 섬으로 수도는 하갓냐이다. 괌의 경제는 대한민국, 일본, 홍콩, 중화민국, 미국, 캐나다 등에서 오는 관광객들을 중심인 관광 사업으로 이뤄진다.

유엔의 식민지 해방 위원회는 괌을 비자치령 목록에 포함하였다. 일본이 괌을 침략했을 때에 일본어로 오미야섬(일본어: 大宮島)이라 불렀다. 현재는 유사시 동아시아를 지원할 해군 시설이 있다. 한반도 유사시 대한민국을 지원하는 미국 제7함대 등이 주둔한다. 괌은 대한민국에서 약 3,000km, 조선민주주의인민공화국에서 약 3,500km 떨어져 있다.

역사

지금의 괌 지역에 남동부 인도네시아에 살던 사람들이 건너와 살았던 기원전 21세기부터 사람이 거주했다.
포르투갈인 항해사 페르디난드 마젤란


In [6]:
# Cell 6: SOURCES 전체 처리 + Document 객체로 변환
documents = []

for source in SOURCES:
    print(f"처리 중: {source['title']}")
    text = clean_wiki_text(source["url"])

    if not text:
        print(f"  → 본문 추출 실패. 건너뜀.")
        continue

    doc = Document(
        page_content=text,
        metadata={
            "source": source["url"],
            "category": source["category"],
            "title": source["title"],
        },
    )
    documents.append(doc)
    print(f"  → {len(text):,} 글자")

print()
print(f"총 {len(documents)}개 Document 생성")
print(f"전체 글자 수: {sum(len(d.page_content) for d in documents):,}")

처리 중: 괌 (한국어 위키백과)
  → 4,812 글자
처리 중: Guam (English Wikipedia)
  → 43,909 글자
처리 중: Guam (Wikivoyage)
  → 27,568 글자

총 3개 Document 생성
전체 글자 수: 76,289


In [7]:
# Cell 7: Wikivoyage를 h2(+특별 h3) 단위로 분리 → 카테고리별 Document 생성

WIKIVOYAGE_SECTION_TO_CATEGORY = {
    "Regions": "명소",
    "Villages": "명소",
    "Other destinations": "명소",
    "See": "명소",
    "Do": "액티비티",
    "Eat": "식당",
    "Drink": "식당",
    "Sleep": "리조트",
    "Get in": "교통",
    "Get around": "교통",
    "Stay safe": "안전·날씨",
    "Stay healthy": "안전·날씨",
}

# h3 특별 분리: 부모 h2 안에 묻혀있는 정보 중 다른 카테고리에 속하는 것
# (현재는 Understand>Climate만 — 시연 질문 2번 "5월 날씨"용)
WIKIVOYAGE_H3_SPECIAL = {
    "Climate": "안전·날씨",
}


def split_wikivoyage_by_section(url: str, section_map: dict, h3_special: dict) -> list[Document]:
    """Wikivoyage 페이지를 h2 단위로 분리.
    - ul/ol/dl 리스트 + raw text table(자손 본문 후보 0개) 본문 포함
    - h3_special에 있는 h3는 별도 섹션으로 분리 (다른 카테고리로 이동)"""
    headers = {"User-Agent": os.environ["USER_AGENT"]}
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()

    soup = bs4.BeautifulSoup(resp.text, "html.parser")
    content = soup.find(id="mw-content-text")
    if content is None:
        return []

    for selector in REMOVE_SELECTORS:
        for tag in content.select(selector):
            tag.decompose()

    def block_text(tag):
        if tag.name in ("ul", "ol"):
            items = []
            for li in tag.find_all("li", recursive=False):
                t = li.get_text(separator="").strip()
                if t:
                    items.append(f"- {t}")
            return "\n".join(items)
        elif tag.name == "dl":
            items = []
            for child in tag.find_all(["dt", "dd"], recursive=False):
                t = child.get_text(separator="").strip()
                if t:
                    items.append(t)
            return "\n".join(items)
        elif tag.name == "table":
            return tag.get_text(separator=" ", strip=True)
        else:  # p, h2, h3, h4
            return tag.get_text(separator="").strip()

    def is_top_level(tag, root):
        for parent in tag.parents:
            if parent is root:
                return True
            if parent.name in ("ul", "ol", "dl", "li", "dt", "dd"):
                return False
        return True

    def is_text_only_table(table):
        for sub in ("p", "ul", "ol", "dl"):
            if table.find(sub) is not None:
                return False
        return True

    candidates = []
    for tag in content.find_all(["h2", "h3", "h4", "p", "ul", "ol", "dl", "table"]):
        if not is_top_level(tag, content):
            continue
        if tag.name == "table" and not is_text_only_table(tag):
            continue
        candidates.append(tag)

    blocks = candidates

    # h2 + 특별 h3 단위로 섹션 분리
    sections = []
    current_title = "Introduction"
    current_category = "general"
    current_blocks = []

    for tag in blocks:
        title_text = tag.get_text(separator="").strip()

        if tag.name == "h2":
            if current_blocks:
                sections.append((current_title, current_category, current_blocks))
            current_title = title_text
            current_category = section_map.get(current_title, "general")
            current_blocks = []
        elif tag.name == "h3" and title_text in h3_special:
            # 특별 h3: 부모 h2 섹션을 닫고 새 섹션 시작
            if current_blocks:
                sections.append((current_title, current_category, current_blocks))
            current_title = title_text
            current_category = h3_special[title_text]
            current_blocks = []
        else:
            text = block_text(tag)
            if text:
                current_blocks.append(text)

    if current_blocks:
        sections.append((current_title, current_category, current_blocks))

    docs = []
    for title, category, body_blocks in sections:
        text = f"{title}\n\n" + "\n\n".join(body_blocks)
        text = re.sub(r"  +", " ", text)
        text = re.sub(r"\(\s+", "(", text)
        text = re.sub(r"\s+\)", ")", text)
        text = re.sub(r"\s+([,;:.])", r"\1", text)

        docs.append(Document(
            page_content=text,
            metadata={
                "source": url,
                "category": category,
                "title": f"Guam Wikivoyage - {title}",
                "section": title,
            },
        ))
    return docs


# 1. Wikivoyage를 섹션별로 분리
wv_url = "https://en.wikivoyage.org/wiki/Guam"
wv_documents = split_wikivoyage_by_section(
    wv_url, WIKIVOYAGE_SECTION_TO_CATEGORY, WIKIVOYAGE_H3_SPECIAL
)

# 2. 기존 documents에서 Wikivoyage 통째 항목 제거 + 섹션별 재추가
documents = [d for d in documents if d.metadata["source"] != wv_url]
documents.extend(wv_documents)

# 3. 검증 출력 — 카테고리별 그룹핑 + 이전 결과 대비 변동
from collections import defaultdict
by_cat = defaultdict(list)
for d in documents:
    by_cat[d.metadata["category"]].append(d)

# 이전 (옵션 2 적용 전, 23개/81,607자) 카테고리별 글자 수 — 회귀 검증용
PREVIOUS_CHARS = {
    "명소": 3077,
    "액티비티": 465,
    "식당": 1288,
    "리조트": 535,
    "교통": 10115,
    "안전·날씨": 2072,
    "general": 64055,
}

print(f"총 {len(documents)}개 Document")
print(f"전체 글자 수: {sum(len(d.page_content) for d in documents):,} (이전 81,607)")
print()
print("=== 카테고리별 Document 분포 (이전 대비 ±변동) ===")
for cat in ["명소", "액티비티", "식당", "리조트", "교통", "안전·날씨", "general"]:
    docs_in_cat = by_cat.get(cat, [])
    total_chars = sum(len(d.page_content) for d in docs_in_cat)
    diff = total_chars - PREVIOUS_CHARS[cat]
    sign = f"{diff:+,}" if diff != 0 else "±0"
    print(f"  [{cat:8}] {len(docs_in_cat):2}개  {total_chars:7,}자  ({sign})")
    for d in docs_in_cat:
        print(f"     - {d.metadata['title']} ({len(d.page_content):,}자)")

총 24개 Document
전체 글자 수: 81,218 (이전 81,607)

=== 카테고리별 Document 분포 (이전 대비 ±변동) ===
  [명소      ]  4개    3,077자  (±0)
     - Guam Wikivoyage - Regions (1,132자)
     - Guam Wikivoyage - Villages (1,146자)
     - Guam Wikivoyage - Other destinations (549자)
     - Guam Wikivoyage - See (250자)
  [액티비티    ]  1개      465자  (±0)
     - Guam Wikivoyage - Do (465자)
  [식당      ]  1개    1,288자  (±0)
     - Guam Wikivoyage - Eat (1,288자)
  [리조트     ]  1개      535자  (±0)
     - Guam Wikivoyage - Sleep (535자)
  [교통      ]  2개   10,115자  (±0)
     - Guam Wikivoyage - Get in (5,360자)
     - Guam Wikivoyage - Get around (4,755자)
  [안전·날씨   ]  3개    2,390자  (+318)
     - Guam Wikivoyage - Climate (318자)
     - Guam Wikivoyage - Stay safe (942자)
     - Guam Wikivoyage - Stay healthy (1,130자)
  [general ] 12개   63,348자  (-707)
     - 괌 (한국어 위키백과) (4,812자)
     - Guam (English Wikipedia) (43,909자)
     - Guam Wikivoyage - Introduction (270자)
     - Guam Wikivoyage - Understand (6,483자)
     - Guam Wikivoyage -

In [8]:
# Cell 8: documents 24개를 data/raw/*.md로 저장
#   - 옵션 A: YAML frontmatter (메타+본문 한 파일에)
#   - 옵션 2: 파일명 카테고리 영문 매핑 (Document.metadata는 한국어 그대로 유지)
import yaml
from pathlib import Path

# 한국어 카테고리 → 영문 파일명 (옵션 2)
CATEGORY_TO_FILENAME = {
    "명소": "attractions",
    "액티비티": "activities",
    "식당": "food",
    "리조트": "resorts",
    "교통": "transport",
    "안전·날씨": "safety",
    "general": "general",
}

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# 재실행 안전: 기존 .md 파일 모두 삭제 (오래된 결과 섞임 방지)
for old in RAW_DIR.glob("*.md"):
    old.unlink()

def make_filename(doc):
    """
    Document → 파일명.
    - Wikivoyage 섹션 분리: {category}_Wikivoyage_{section}.md
    - 한국어 위키:           general_Wikipedia_ko.md
    - 영문 위키:             general_Wikipedia_en.md
    """
    meta = doc.metadata
    category_en = CATEGORY_TO_FILENAME[meta["category"]]
    src_url = meta.get("source", "").lower()

    if "wikivoyage" in src_url:
        section_safe = meta["section"].replace(" ", "_")
        return f"{category_en}_Wikivoyage_{section_safe}.md"
    elif "ko.wikipedia" in src_url:
        return f"{category_en}_Wikipedia_ko.md"
    elif "en.wikipedia" in src_url:
        return f"{category_en}_Wikipedia_en.md"
    else:
        raise ValueError(f"Unknown source URL: {meta.get('source')}")

# 저장 + 검증
saved = []
for doc in documents:
    fname = make_filename(doc)
    fpath = RAW_DIR / fname

    # YAML frontmatter — allow_unicode=True 로 한국어 그대로 (escape 안 함)
    fm = yaml.safe_dump(doc.metadata, allow_unicode=True, sort_keys=False)
    fpath.write_text(f"---\n{fm}---\n\n{doc.page_content}\n", encoding="utf-8")
    saved.append((fname, len(doc.page_content)))

# 출력
print(f"저장 완료: {len(saved)}개 파일 → {RAW_DIR.resolve()}")
print(f"{'파일명':<55} {'본문자수':>8}")
print("-" * 70)
for fname, size in sorted(saved):
    print(f"{fname:<55} {size:>8,}")

# 회귀 체크
assert len(saved) == 24, f"파일 수 불일치: {len(saved)} (기대 24)"
assert len(set(f for f, _ in saved)) == 24, "파일명 중복 발생"
total = sum(s for _, s in saved)
assert total == 81218, f"총 본문 자수 불일치: {total} (기대 81218)"
print(f"\n✓ 검증 통과: 24개 / 파일명 중복 없음 / 총 본문 {total:,}자 (Cell 7 결과와 일치)")

저장 완료: 24개 파일 → C:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\data\raw
파일명                                                         본문자수
----------------------------------------------------------------------
activities_Wikivoyage_Do.md                                  465
attractions_Wikivoyage_Other_destinations.md                 549
attractions_Wikivoyage_Regions.md                          1,132
attractions_Wikivoyage_See.md                                250
attractions_Wikivoyage_Villages.md                         1,146
food_Wikivoyage_Eat.md                                     1,288
general_Wikipedia_en.md                                   43,909
general_Wikipedia_ko.md                                    4,812
general_Wikivoyage_Buy.md                                  1,965
general_Wikivoyage_Connect.md                              2,956
general_Wikivoyage_Cope.md                                   744
general_Wikivoyage_Go_next.md                                12

## PIC 자료 보강 — 나무위키 "퍼시픽 아일랜드 클럽" (2026-05-07)

위키백과(ko/en)·Wikivoyage 24개 자료에 PIC 정보가 0건이라 시연 Q1("PIC 리조트 어때?") 답변이 부실. 나무위키 PIC 페이지 1개를 추가 보강.

- **라이선스**: CC BY-NC-SA 2.0 KR — 비상업·학술 맥락에서 사용 가능. 출처 URL을 메타에 명시
- **robots.txt**: `/w/` 경로 허용 — 정책 위반 없음
- **진행 흐름**: ① requests로 자동 fetch 시도 → ② SPA 확인 후 수동 복사로 전환 → ③ 광고·잡음 정제 → ④ frontmatter + .md 저장

### 1단계: 페이지 받기 시도 (requests + 브라우저 UA 헤더)

봇 차단 우회를 위해 평범한 크롬 User-Agent로 위장하고 페이지를 가져오기 시도.

**판정 기준**: 응답 코드 200 + 본문에 PIC 키워드 포함 → 자동 fetch 가능. 빈 껍데기 또는 403 → 수동 복사로 전환.

**실측 결과**: 응답 200 / 99,989자 수신 / PIC 키워드 포함 — 1차 통과

In [9]:
import requests

URL = "https://namu.wiki/w/퍼시픽 아일랜드 클럽"

# "나는 봇 아니고 크롬 브라우저예요" 자기소개 헤더
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

response = requests.get(URL, headers=HEADERS, timeout=10)

print(f"응답 코드     : {response.status_code}")
print(f"받은 글자 수  : {len(response.text):,}자")

# 받은 HTML 안에 PIC 본문 키워드가 들어 있는지
has_keyword = ("퍼시픽 아일랜드 클럽" in response.text 
               or "Pacific Islands Club" in response.text)
print(f"PIC 키워드 포함: {has_keyword}")

print(f"\n--- HTML 앞 500자 미리보기 ---\n{response.text[:500]}")

응답 코드     : 200
받은 글자 수  : 99,989자
PIC 키워드 포함: True

--- HTML 앞 500자 미리보기 ---
<!doctype html>
<html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="user-scalable=no, initial-scale=1, width=device-width, viewport-fit=cover">
<title>퍼시픽 아일랜드 클럽 - 나무위키</title>
<meta http-equv="x-ua-compatible" content="ie=edge">
<meta name="generator" content="the seed">
<meta name="mobile-web-app-capable" content="yes">
<meta name="application-name" content="나무위키">
<meta name="msapplication-tooltip" content="나무위키">
<meta name="color-scheme" content="light dark">
<meta 


### 2단계: 본문 영역 진단 (BeautifulSoup 셀렉터 매칭)

본문 컨테이너 후보 셀렉터(`.wiki-paragraph`, `article`, `main` 등)를 한꺼번에 시도해서 어느 게 본문을 잡는지 측정.

**실측 결과**:
- 모든 후보 셀렉터 매칭 실패
- body 전체 텍스트 3,762자뿐 (받은 HTML 10만 자 대비 비정상적으로 적음)
- "Pacific Islands Club" 키워드 위치는 `<meta property="og:description">` 안 — SNS 공유용 메타태그에만 박혀있고 본문 영역은 비어있음

**결론**: 나무위키는 SPA(자바스크립트 렌더링) 구조. requests 한 번으로는 본문 추출 불가능 → **수동 복사로 전환**

In [10]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

# 나무위키 본문이 들어있을 만한 후보 영역들
candidates = [
    ".wiki-paragraph",
    ".wiki-content",
    "article",
    "main",
    "#article",
    ".article",
]

print("=== 셀렉터별 매칭 결과 ===")
for sel in candidates:
    found = soup.select(sel)
    if found:
        total_chars = sum(len(el.get_text(strip=True)) for el in found)
        print(f"  {sel:20s} → {len(found):3d}개 요소, 총 {total_chars:,}자")
    else:
        print(f"  {sel:20s} → 없음")

# 보너스 정보
title_tag = soup.select_one("title")
print(f"\n페이지 title: {title_tag.get_text() if title_tag else '없음'}")

body = soup.select_one("body")
if body:
    print(f"body 전체 텍스트: {len(body.get_text()):,}자")

# PIC 핵심 문구가 HTML 어디에 박혀있는지
needle = "Pacific Islands Club"
if needle in response.text:
    pos = response.text.index(needle)
    print(f"\n'{needle}' HTML 위치: {pos:,}번째 글자")
    print(f"주변 300자 (앞 100 + 본문 시작 200):")
    print(f"  ...{response.text[max(0, pos-100):pos+200]}...")

=== 셀렉터별 매칭 결과 ===
  .wiki-paragraph      → 없음
  .wiki-content        → 없음
  article              → 없음
  main                 → 없음
  #article             → 없음
  .article             → 없음

페이지 title: 퍼시픽 아일랜드 클럽 - 나무위키
body 전체 텍스트: 3,762자

'Pacific Islands Club' HTML 위치: 999번째 글자
주변 300자 (앞 100 + 본문 시작 200):
  ...x1OLatE6RE4WleTxMi5j_mJLjmcz5hMPBZ7HLkKY1pn0QrmHsyww.svg">
<meta property="og:description" content="Pacific Islands Club, PIC 괌  및  북마리아나 제도 에 위치한 미국의 복합형 리조트. 흔히  ">
<meta property="og:url" content="https://namu.wiki/w/%ED%8D%BC%EC%8B%9C%ED%94%BD%20%EC%95%84%EC%9D%BC%EB%9E%9C%EB%93%9C%20%ED%81%B4%E...


### 3단계: 수동 복사본 정제

브라우저로 PIC 페이지의 본문 영역을 복사해 `practice/pic_temp.txt`로 저장 (UTF-8, 2,617자).

**정제 대상**:
- `[편집]` 표시 (모든 헤딩에 붙어있음)
- `resort-g188a6d9f...` 트립어드바이저 ID 잔재 한 줄
- 본문 끝부분에 끼어든 사주·타로·점 광고 영역 (`CC-white 이 문단의 내용...` 시스템 메시지부터 `[1]` 각주 직전까지) — 약 16줄

**보존 대상**:
- 각주 [1]~[6] (PIC 역사·한국지사·야간 비행 도착 후 식사 패턴 등 가치 있는 정보)

**결과**: 2,617자 → 2,215자 (광고 + `[편집]` 등 402자 제거)

In [11]:
import re
from pathlib import Path

# 임시 파일 위치 (위치 정해진 후 채움)
TEMP_PATH = Path("pic_temp.txt")   # ← 여기 사용자 결정 따라 수정

raw = TEMP_PATH.read_text(encoding="utf-8")
print(f"원본 길이: {len(raw):,}자")

# 1. "[편집]" 제거
text = raw.replace("[편집]", "")

# 2. "resort-g..." 같은 트립어드바이저 잔재 한 줄 제거
text = re.sub(r"^resort-[a-z0-9]+\.\.\.\s*\n", "", text, flags=re.MULTILINE)

# 3. 광고 영역 제거: "CC-white 이 문단의 내용..." 시작 ~ "[1] " 직전까지 통째로
ad_pattern = re.compile(
    r"CC-white\s+이\s+문단의\s+내용.+?(?=\[1\]\s)",
    flags=re.DOTALL,
)
text = ad_pattern.sub("", text)

# 4. 연속 빈줄 3개 이상 → 2개로
text = re.sub(r"\n{3,}", "\n\n", text)

# 5. 앞뒤 공백 정리
text = text.strip()

print(f"정제 후 길이: {len(text):,}자  (원본 대비 -{len(raw)-len(text):,}자)")
print("\n" + "="*60)
print("정제된 본문 첫 400자")
print("="*60)
print(text[:400])
print("\n" + "="*60)
print("정제된 본문 마지막 400자")
print("="*60)
print(text[-400:])

원본 길이: 2,617자
정제 후 길이: 2,215자  (원본 대비 -402자)

정제된 본문 첫 400자
1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불린다. 괌 및 사이판에 여행하는 관광객들이 가장 많이 찾는 숙박시설이다.

한국인, 일본인 관광객들이 가장 많이 숙박하며, 각 리조트에는 한국인, 일본인 직원 및 현지인 중 한국어 및 일본어 구사가 가능한 직원이 배치되어있다.

워낙 한국인 관광객이 많이 찾다 보니, 한국어 사이트도 개설되어 있다.[5]
2. PIC 괌
괌 투몬만에 위치한 퍼시픽 아일랜드 클럽 괌
PIC 하면 생각나는 리조트이며, PIC 리조트 중 가장 큰 리조트이다. 괌에서 최대 규모를 자랑하며, 괌을 찾는 관광객들이 가장 많이 애용한다. 특히 가족 단위 관광객들에게 특화되어 있으며 다양한 레크리에

정제된 본문 마지막 400자
0년부터 운영도 직영방식으로 운영중이다. 현재 PIC사이판의 경우 켄싱턴호텔앤리조트가 Premier Hotels & Resorts사에 경영을 위탁하여 운영하는 방식으로 계약되어있다.
[5] 다른 외국 호텔 사이트의 한국어 버전 사이트와 달리 발번역 없이 깔끔하고 한국 웹사이트처럼 깔끔한 디자인으로 구성되어 있다. 괌PIC의 실소유자이자 사이판PIC의 위탁운영사인 프리미어호텔앤드리조트 그룹의 한국지사에서 직접 홍보와 사이트 운영을 담당하다 보니 괌의 타 호텔과 달리 한국에서도 호텔에 대한 많은 공식정보를 얻을 수 있다.
[6] 한국발 괌 비행기가 야간에 많이 운항하기 때문에, 새벽 2~3시에 호텔에 도착하는 한국인 관광객은 늦잠 자느라 조식은 생략하는 편이며, 호텔에서의 첫 식사는 그날 석식때 하기 때문이다.


### 4단계: frontmatter 메타 + .md 저장 (다른 24개와 같은 형식)

저장 위치: `data/raw/resorts_PIC_namuwiki.md`

**메타 3개** (02_build_index.ipynb 검증 기준과 호환):
- `title`: "퍼시픽 아일랜드 클럽 (PIC) - 나무위키"
- `source`: 나무위키 페이지 URL
- `category`: "리조트"

**즉시 호환성 검증**: 저장 직후 02_build_index.ipynb의 `load_md_with_frontmatter` 함수로 다시 읽어서 frontmatter 파싱 + 필수 키 체크.

In [12]:
import yaml
from pathlib import Path

# 1. frontmatter 메타 (다른 24개 .md와 같은 형식)
metadata_yaml = """---
title: "퍼시픽 아일랜드 클럽 (PIC) - 나무위키"
source: "https://namu.wiki/w/퍼시픽 아일랜드 클럽"
category: "리조트"
---
"""

# 2. 메타 + 정제된 본문 합쳐서 .md 파일 작성
final_md = metadata_yaml + text   # text는 직전 셀의 정제 결과

# 3. 다른 24개 .md와 같은 폴더에 저장
OUTPUT_PATH = Path("../data/raw/resorts_PIC_namuwiki.md")
OUTPUT_PATH.write_text(final_md, encoding="utf-8")

print(f"✓ 저장 완료: {OUTPUT_PATH.resolve()}")
print(f"  파일 크기: {OUTPUT_PATH.stat().st_size:,} bytes")

# 4. 호환성 검증 — 02_build_index.ipynb의 frontmatter 파서로 다시 읽어보기
def load_md_with_frontmatter(fpath):
    txt = Path(fpath).read_text(encoding="utf-8")
    if not txt.startswith("---\n"):
        raise ValueError(f"frontmatter 없음: {fpath.name}")
    _, fm, body = txt.split("---\n", 2)
    metadata = yaml.safe_load(fm)
    return metadata, body.strip()

meta, body = load_md_with_frontmatter(OUTPUT_PATH)
print(f"\n--- 메타데이터 검증 ---")
print(f"  title:    {meta.get('title')}")
print(f"  source:   {meta.get('source')}")
print(f"  category: {meta.get('category')}")
print(f"  본문 글자 수: {len(body):,}자")

# 5. 필수 키 체크 (02_build_index.ipynb 셀 7과 같은 검증)
required = {"source", "category", "title"}
missing = required - set(meta.keys())
if missing:
    print(f"  ⚠️ 누락된 필수 메타: {missing}")
else:
    print(f"  ✓ 필수 메타 3개(source/category/title) 모두 있음")

✓ 저장 완료: C:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\data\raw\resorts_PIC_namuwiki.md
  파일 크기: 5,075 bytes

--- 메타데이터 검증 ---
  title:    퍼시픽 아일랜드 클럽 (PIC) - 나무위키
  source:   https://namu.wiki/w/퍼시픽 아일랜드 클럽
  category: 리조트
  본문 글자 수: 2,215자
  ✓ 필수 메타 3개(source/category/title) 모두 있음
